# Forams DINOv2-B controlled experiment

This notebook keeps the successful ResNet101 centers fixed and tests whether DINOv2-B improves class assignment. The 18 grayscale slices for each candidate are grouped into six adjacent-slice RGB composites. Training uses the packed-simulation cache, official attached DINOv2 weights, best-epoch selection, and conservative pseudo-label adaptation. It writes standalone, blended, and conservative submissions; `submission.csv` is the conservative variant.


In [ ]:
import gc
import hashlib
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel


SEED = 20260821
LABELS = [f"cl{i:02d}" for i in range(14)]
INPUT_ROOT = Path("/kaggle/input")
CACHE_ROOT = INPUT_ROOT / "notebooks" / "hanifnoerrofiq" / "forams-packed-sim-cache-cpu"
TEST_ROOT = INPUT_ROOT / "competitions" / "forams-2026" / "test"
WORK_DIR = Path("/kaggle/working")
TEMP_DIR = Path("/kaggle/temp/forams_dinov2b")

IMAGE_SIZE = 224
COMPOSITES_PER_OBJECT = 6
EPOCHS = 5
BATCH_SIZE = 32
CANDIDATES_PER_BATCH = 4
NUM_WORKERS = 4
HEAD_LEARNING_RATE = 3e-4
FINETUNE_HEAD_LEARNING_RATE = 1e-4
BACKBONE_LEARNING_RATE = 1e-5
ADAPTATION_HEAD_LEARNING_RATE = 5e-5
ADAPTATION_BACKBONE_LEARNING_RATE = 5e-6
WEIGHT_DECAY = 2e-4
PSEUDO_WEIGHT = 0.20
BLEND_DINO_WEIGHT = 0.35
CHANGE_CONFIDENCE = 0.72
CHANGE_MARGIN = 0.25
MAXIMUM_CHANGED_FRACTION = 0.12

IMAGENET_MEAN = torch.tensor((0.485, 0.456, 0.406))[:, None, None]
IMAGENET_STD = torch.tensor((0.229, 0.224, 0.225))[:, None, None]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def log(message):
    print(message, flush=True)


def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


def require_2xt4():
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    log(f"[runtime] torch={torch.__version__} cuda={names}")
    if len(names) != 2 or not all("T4" in name.upper() for name in names):
        raise RuntimeError(f"This notebook requires exactly 2xT4, found {names}")


def find_one(root, filename):
    matches = sorted({path.resolve() for path in root.rglob(filename) if path.is_file()})
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {filename} below {root}, found {[str(path) for path in matches]}"
        )
    return matches[0]


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_dinov2_root():
    preferred = [
        INPUT_ROOT / "models" / "metaresearch" / "dinov2" / "pytorch" / "base" / "1",
        INPUT_ROOT / "dinov2" / "pytorch" / "base" / "1",
    ]
    candidates = [path for path in preferred if (path / "config.json").is_file()]
    if not candidates:
        for config_path in INPUT_ROOT.rglob("config.json"):
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if config.get("model_type") == "dinov2":
                candidates.append(config_path.parent)
    candidates = sorted({path.resolve() for path in candidates})
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one attached DINOv2 model, found {[str(path) for path in candidates]}"
        )
    model_root = candidates[0]
    weight_files = list(model_root.glob("*.safetensors")) + list(model_root.glob("*.bin"))
    if not weight_files:
        raise FileNotFoundError(f"No DINOv2 weights found in {model_root}")
    log(f"[dinov2] root={model_root} weights={[path.name for path in weight_files]}")
    return model_root


def load_inputs():
    manifest_path = find_one(CACHE_ROOT, "packed_sim_cache_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    cache_files = {
        key: find_one(CACHE_ROOT, filename)
        for key, filename in manifest["files"].items()
    }
    train_views = np.load(cache_files["train_views"], mmap_mode="r")
    validation_views = np.load(cache_files["validation_views"], mmap_mode="r")
    if train_views.shape[1:] != (18, 128, 128):
        raise ValueError(f"Unexpected train view shape: {train_views.shape}")
    if validation_views.shape[1:] != (18, 128, 128):
        raise ValueError(f"Unexpected validation view shape: {validation_views.shape}")

    candidate_path = find_one(INPUT_ROOT, "resnet101_candidates.csv")
    candidates = pd.read_csv(candidate_path)
    required = {
        "filename", "candidate_rank", "d0", "d1", "d2", "resnet101_final_label"
    } | {f"resnet101_p{i:02d}" for i in range(14)}
    missing = sorted(required - set(candidates.columns))
    if missing:
        raise ValueError(f"Candidate artifact is missing columns: {missing}")
    candidates = candidates.sort_values(["filename", "candidate_rank"]).reset_index(drop=True)
    candidates["row_index"] = np.arange(len(candidates))
    if len(candidates) != 2425 or candidates["filename"].nunique() != 95:
        raise ValueError(
            f"Expected 2,425 candidates across 95 volumes, found {len(candidates)} "
            f"across {candidates['filename'].nunique()}"
        )
    coordinates = candidates[["d0", "d1", "d2"]].to_numpy(dtype=np.float64)
    if not np.isfinite(coordinates).all():
        raise ValueError("Candidate coordinates contain non-finite values")
    log(
        f"[inputs] train={train_views.shape} validation={validation_views.shape} "
        f"candidates={len(candidates)} candidate_sha256={sha256(candidate_path)}"
    )
    return {
        "train_views": cache_files["train_views"],
        "train_labels": cache_files["train_labels"],
        "validation_views": cache_files["validation_views"],
        "validation_labels": cache_files["validation_labels"],
        "candidates": candidates,
        "candidate_path": candidate_path,
        "candidate_sha256": sha256(candidate_path),
    }


def crop_plane(plane, center, size):
    cy, cx = np.rint(center).astype(int)
    half = size // 2
    output = np.full((size, size), int(np.median(plane)), dtype=np.uint8)
    sy0 = max(0, cy - half)
    sy1 = min(plane.shape[0], cy - half + size)
    sx0 = max(0, cx - half)
    sx1 = min(plane.shape[1], cx - half + size)
    ty0 = sy0 - (cy - half)
    tx0 = sx0 - (cx - half)
    output[ty0 : ty0 + sy1 - sy0, tx0 : tx0 + sx1 - sx0] = plane[sy0:sy1, sx0:sx1]
    return output


def normalize_resize(patch, size=128):
    sample = patch[::2, ::2]
    low = min(float(np.percentile(sample, 5.0)), 80.0)
    high = max(float(np.percentile(sample, 99.5)), low + 32.0)
    normalized = np.clip((patch.astype(np.float32) - low) / (high - low), 0.0, 1.0)
    image = Image.fromarray(np.rint(normalized * 255.0).astype(np.uint8))
    image = image.resize((size, size), resample=Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.uint8)


def extract_views(volume, center):
    views = []
    for crop_size in (160, 256):
        for axis in range(3):
            remaining = [dimension for dimension in range(3) if dimension != axis]
            for offset in (-24, 0, 24):
                index = int(np.clip(round(center[axis] + offset), 0, volume.shape[axis] - 1))
                plane = volume[index] if axis == 0 else volume[:, index] if axis == 1 else volume[:, :, index]
                patch = crop_plane(
                    plane,
                    (float(center[remaining[0]]), float(center[remaining[1]])),
                    crop_size,
                )
                views.append(normalize_resize(patch))
    return np.stack(views).astype(np.uint8)


def build_real_view_cache(candidates):
    output_path = TEMP_DIR / "real_candidate_multiview.npy"
    views = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=np.uint8, shape=(len(candidates), 18, 128, 128)
    )
    started = time.time()
    written = 0
    for volume_index, (filename, subset) in enumerate(
        candidates.groupby("filename", sort=True), start=1
    ):
        volume = tifffile.imread(TEST_ROOT / filename)
        for row in subset.sort_values("row_index").itertuples(index=False):
            center = np.asarray([row.d0, row.d1, row.d2], dtype=np.float32)
            views[int(row.row_index)] = extract_views(volume, center)
            written += 1
        views.flush()
        del volume
        gc.collect()
        if volume_index % 10 == 0 or volume_index == 95:
            log(
                f"[real-views] volumes={volume_index}/95 candidates={written}/{len(candidates)} "
                f"elapsed_min={(time.time() - started) / 60:.1f}"
            )
    del views
    return output_path


def normalize_image(image):
    return (image - IMAGENET_MEAN) / IMAGENET_STD


class CompositeDataset(Dataset):
    def __init__(self, views_path, labels, train, sample_weight=1.0, pseudo=False):
        self.views = np.load(views_path, mmap_mode="r")
        self.labels = np.load(labels, mmap_mode="r") if isinstance(labels, Path) else labels
        self.train = train
        self.sample_weight = float(sample_weight)
        self.pseudo = pseudo

    def __len__(self):
        return len(self.labels) * COMPOSITES_PER_OBJECT

    def __getitem__(self, index):
        sample = index // COMPOSITES_PER_OBJECT
        composite = index % COMPOSITES_PER_OBJECT
        start = composite * 3
        image = torch.from_numpy(
            np.array(self.views[sample, start : start + 3], copy=True)
        ).float() / 255.0
        if self.train:
            image = torch.rot90(image, random.randrange(4), dims=(-2, -1))
            if random.random() < 0.5:
                image = torch.flip(image, dims=(-1,))
            if random.random() < 0.5:
                image = torch.flip(image, dims=(-2,))
            if random.random() < 0.5:
                image = torch.flip(image, dims=(0,))
            gamma = random.uniform(0.92, 1.10) if self.pseudo else random.uniform(0.86, 1.16)
            noise = random.uniform(0.0, 0.010) if self.pseudo else random.uniform(0.0, 0.016)
            image = torch.clamp(image.pow(gamma) + torch.randn_like(image) * noise, 0.0, 1.0)
        image = F.interpolate(
            image[None], size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
        )[0]
        return normalize_image(image), int(self.labels[sample]), self.sample_weight


class DinoClassifier(nn.Module):
    def __init__(self, model_root):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(str(model_root), local_files_only=True)
        hidden_size = int(self.backbone.config.hidden_size)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.25),
            nn.Linear(hidden_size, 14),
        )

    def forward(self, images):
        output = self.backbone(pixel_values=images)
        return self.head(output.pooler_output)


def freeze_backbone(model):
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True


def unfreeze_last_blocks(model, block_count=2):
    freeze_backbone(model)
    layers = model.backbone.encoder.layer
    for layer in layers[-block_count:]:
        for parameter in layer.parameters():
            parameter.requires_grad = True
    for parameter in model.backbone.layernorm.parameters():
        parameter.requires_grad = True


def parallel(model):
    model = model.to(device)
    return nn.DataParallel(model) if torch.cuda.device_count() > 1 else model


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def clone_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def checkpoint_is_better(accuracy, loss, best_accuracy, best_loss):
    return accuracy > best_accuracy + 1e-12 or (
        abs(accuracy - best_accuracy) <= 1e-12 and loss < best_loss
    )


def selected_epoch(history):
    best_accuracy = max(row["accuracy"] for row in history)
    tied = [row for row in history if abs(row["accuracy"] - best_accuracy) <= 1e-12]
    return min(tied, key=lambda row: row["validation_loss"])


def optimizer_for(model, head_only):
    bare = unwrap(model)
    if head_only:
        return torch.optim.AdamW(
            bare.head.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
    backbone = [parameter for parameter in bare.backbone.parameters() if parameter.requires_grad]
    return torch.optim.AdamW(
        [
            {"params": backbone, "lr": BACKBONE_LEARNING_RATE},
            {"params": bare.head.parameters(), "lr": FINETUNE_HEAD_LEARNING_RATE},
        ],
        weight_decay=WEIGHT_DECAY,
    )


@torch.inference_mode()
def predict_candidates(model, views, candidates_per_batch=CANDIDATES_PER_BATCH):
    model.eval()
    outputs = []
    mean = IMAGENET_MEAN[None].to(device)
    std = IMAGENET_STD[None].to(device)
    for start in range(0, len(views), candidates_per_batch):
        raw = torch.from_numpy(
            np.array(views[start : start + candidates_per_batch], copy=True)
        ).float() / 255.0
        batch = len(raw)
        images = raw.reshape(batch, COMPOSITES_PER_OBJECT, 3, 128, 128)
        images = images.reshape(batch * COMPOSITES_PER_OBJECT, 3, 128, 128).to(
            device, non_blocking=True
        )
        images = F.interpolate(
            images, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
        )
        images = (images - mean) / std
        with torch.amp.autocast("cuda"):
            logits = model(images).reshape(batch, COMPOSITES_PER_OBJECT, 14)
        outputs.append(torch.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
    return np.concatenate(outputs)


def validation_metrics(model, views_path, labels_path):
    views = np.load(views_path, mmap_mode="r")
    labels = np.load(labels_path)
    probabilities = predict_candidates(model, views)
    predictions = probabilities.argmax(axis=1)
    validation_loss = -np.log(
        np.clip(probabilities[np.arange(len(labels)), labels], 1e-7, 1.0)
    ).mean()
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "mean_confidence": float(probabilities.max(axis=1).mean()),
        "validation_loss": float(validation_loss),
    }


def train_supervised(model, inputs):
    dataset = CompositeDataset(inputs["train_views"], inputs["train_labels"], train=True)
    labels = np.load(inputs["train_labels"])
    expanded_labels = np.repeat(labels, COMPOSITES_PER_OBJECT)
    counts = np.bincount(expanded_labels, minlength=14)
    sampler = WeightedRandomSampler(
        1.0 / np.maximum(counts[expanded_labels], 1), len(dataset), replacement=True
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )
    freeze_backbone(model)
    model = parallel(model)
    optimizer = optimizer_for(model, head_only=True)
    scaler = torch.amp.GradScaler("cuda")
    history = []
    best_state = None
    best_accuracy = -1.0
    best_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        if epoch == 2:
            unfreeze_last_blocks(unwrap(model), block_count=2)
            optimizer = optimizer_for(model, head_only=False)
            log("[train] unfroze final two DINOv2 transformer blocks")
        model.train()
        losses = []
        for images, targets, weights in loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            weights = weights.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                per_item = F.cross_entropy(
                    model(images), targets, reduction="none", label_smoothing=0.04
                )
                loss = (per_item * weights).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [parameter for parameter in model.parameters() if parameter.requires_grad], 2.0
            )
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        metrics = validation_metrics(
            model, inputs["validation_views"], inputs["validation_labels"]
        )
        metrics.update({"epoch": epoch, "loss": float(np.mean(losses))})
        history.append(metrics)
        log(
            f"[train] epoch={epoch} loss={metrics['loss']:.4f} "
            f"val_loss={metrics['validation_loss']:.4f} "
            f"val_acc={metrics['accuracy']:.4f} val_f1={metrics['macro_f1']:.4f}"
        )
        if checkpoint_is_better(
            metrics["accuracy"], metrics["validation_loss"], best_accuracy, best_loss
        ):
            best_accuracy = metrics["accuracy"]
            best_loss = metrics["validation_loss"]
            best_state = clone_state(unwrap(model))

    bare = unwrap(model)
    bare.load_state_dict(best_state)
    log(f"[selected] {json.dumps(selected_epoch(history))}")
    return bare, history


def build_pseudo_cache(candidates, real_views_path, dino_probabilities):
    anchor_probabilities = candidates[
        [f"resnet101_p{i:02d}" for i in range(14)]
    ].to_numpy(dtype=np.float32)
    anchor_labels = anchor_probabilities.argmax(axis=1)
    dino_labels = dino_probabilities.argmax(axis=1)
    stable = (
        (anchor_labels == dino_labels)
        & (anchor_probabilities.max(axis=1) >= 0.75)
        & (dino_probabilities.max(axis=1) >= 0.72)
    )
    indices = np.flatnonzero(stable)
    pseudo_labels = dino_labels[indices].astype(np.int64)
    source = np.load(real_views_path, mmap_mode="r")
    output_path = TEMP_DIR / "stable_real_views.npy"
    if len(indices) == 0:
        np.save(output_path, np.empty((0, 18, 128, 128), dtype=np.uint8))
        log("[pseudo] stable=0")
        return output_path, pseudo_labels
    output = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=np.uint8, shape=(len(indices), 18, 128, 128)
    )
    output[:] = source[indices]
    output.flush()
    del output, source
    log(f"[pseudo] stable={len(indices)}")
    return output_path, pseudo_labels


def adapt_if_better(model, inputs):
    if len(inputs["pseudo_labels"]) == 0:
        metrics = validation_metrics(
            model, inputs["validation_views"], inputs["validation_labels"]
        )
        return model, metrics, metrics, False
    before = validation_metrics(model, inputs["validation_views"], inputs["validation_labels"])
    before_state = clone_state(model)
    packed = CompositeDataset(inputs["train_views"], inputs["train_labels"], train=True)
    pseudo = CompositeDataset(
        inputs["pseudo_views"], inputs["pseudo_labels"], train=True,
        sample_weight=PSEUDO_WEIGHT, pseudo=True
    )
    loader = DataLoader(
        ConcatDataset([packed, pseudo]), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
    )
    unfreeze_last_blocks(model, block_count=2)
    model = parallel(model)
    bare = unwrap(model)
    backbone = [parameter for parameter in bare.backbone.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone, "lr": ADAPTATION_BACKBONE_LEARNING_RATE},
            {"params": bare.head.parameters(), "lr": ADAPTATION_HEAD_LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY
    )
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    losses = []
    for images, targets, weights in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            per_item = F.cross_entropy(
                model(images), targets, reduction="none", label_smoothing=0.03
            )
            loss = (per_item * weights).mean()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    bare = unwrap(model)
    after = validation_metrics(bare, inputs["validation_views"], inputs["validation_labels"])
    keep = checkpoint_is_better(
        after["accuracy"], after["validation_loss"], before["accuracy"], before["validation_loss"]
    )
    if not keep:
        bare.load_state_dict(before_state)
    log(f"[adapt] loss={np.mean(losses):.4f} kept={keep} before={before} after={after}")
    return bare, before, after, keep


def build_outputs(candidates, dino_probabilities):
    anchor_probabilities = candidates[
        [f"resnet101_p{i:02d}" for i in range(14)]
    ].to_numpy(dtype=np.float32)
    anchor_probabilities /= np.maximum(anchor_probabilities.sum(axis=1, keepdims=True), 1e-8)
    anchor_labels = candidates["resnet101_final_label"].to_numpy(dtype=int)
    dino_labels = dino_probabilities.argmax(axis=1)
    dino_confidence = dino_probabilities.max(axis=1)
    ordered = np.sort(dino_probabilities, axis=1)
    dino_margin = ordered[:, -1] - ordered[:, -2]

    blended = (1.0 - BLEND_DINO_WEIGHT) * anchor_probabilities + BLEND_DINO_WEIGHT * dino_probabilities
    blended /= blended.sum(axis=1, keepdims=True)
    blend_labels = blended.argmax(axis=1)

    changed = (
        (dino_labels != anchor_labels)
        & (dino_confidence >= CHANGE_CONFIDENCE)
        & (dino_margin >= CHANGE_MARGIN)
        & (anchor_probabilities.max(axis=1) <= 0.82)
    )
    maximum_changes = int(round(MAXIMUM_CHANGED_FRACTION * len(candidates)))
    if changed.sum() > maximum_changes:
        priority = np.where(changed, dino_confidence * dino_margin, -1.0)
        keep = np.argsort(priority)[::-1][:maximum_changes]
        changed[:] = False
        changed[keep] = True
    conservative_labels = np.where(changed, dino_labels, anchor_labels)

    output = candidates.copy()
    output["dinov2_label"] = dino_labels
    output["dinov2_confidence"] = dino_confidence
    output["dinov2_margin"] = dino_margin
    output["dinov2_blend_label"] = blend_labels
    output["dinov2_changed"] = changed
    output["dinov2_final_label"] = conservative_labels
    for class_index in range(14):
        output[f"dinov2_p{class_index:02d}"] = dino_probabilities[:, class_index]
        output[f"dinov2_blend_p{class_index:02d}"] = blended[:, class_index]
    return output


def write_submission(candidates, label_column, output_path):
    rows = []
    for filename, subset in candidates.groupby("filename", sort=True):
        tokens = []
        for row in subset.sort_values("candidate_rank").itertuples(index=False):
            tokens.extend([
                LABELS[int(getattr(row, label_column))],
                f"{row.d0:.2f}", f"{row.d1:.2f}", f"{row.d2:.2f}",
            ])
        rows.append({"filename": filename, "centerpoint": ";".join(tokens)})
    submission = pd.DataFrame(rows)
    if len(submission) != 95 or submission["centerpoint"].eq("").any():
        raise ValueError("Submission must contain 95 non-empty rows")
    submission.to_csv(output_path, index=False)


def run():
    started = time.time()
    seed_everything()
    require_2xt4()
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    model_root = locate_dinov2_root()
    inputs = load_inputs()
    candidates = inputs["candidates"]

    model = DinoClassifier(model_root)
    real_views_path = build_real_view_cache(candidates)
    inputs["real_views"] = np.load(real_views_path, mmap_mode="r")
    model, history = train_supervised(model, inputs)
    initial_probabilities = predict_candidates(model, inputs["real_views"])
    pseudo_views_path, pseudo_labels = build_pseudo_cache(
        candidates, real_views_path, initial_probabilities
    )
    inputs["pseudo_views"] = pseudo_views_path
    inputs["pseudo_labels"] = pseudo_labels
    model, before_adaptation, after_adaptation, adaptation_kept = adapt_if_better(model, inputs)

    probabilities = predict_candidates(model, inputs["real_views"])
    output = build_outputs(candidates, probabilities)
    output.to_csv(WORK_DIR / "dinov2_candidates.csv", index=False)
    write_submission(output, "dinov2_label", WORK_DIR / "submission_dinov2_standalone.csv")
    write_submission(output, "dinov2_blend_label", WORK_DIR / "submission_dinov2_blend.csv")
    write_submission(output, "dinov2_final_label", WORK_DIR / "submission_dinov2_conservative.csv")
    write_submission(output, "dinov2_final_label", WORK_DIR / "submission.csv")
    torch.save(model.cpu().state_dict(), WORK_DIR / "dinov2b_forams.pt")

    summary = {
        "model": "metaresearch/dinov2 PyTorch base/1",
        "candidate_source": str(inputs["candidate_path"]),
        "candidate_sha256": inputs["candidate_sha256"],
        "training_history": history,
        "selected_epoch": selected_epoch(history),
        "before_adaptation": before_adaptation,
        "after_adaptation": after_adaptation,
        "adaptation_kept": adaptation_kept,
        "pseudo_count": int(len(pseudo_labels)),
        "standalone_disagreement_count": int(
            (output["dinov2_label"] != output["resnet101_final_label"]).sum()
        ),
        "blend_changed_count": int(
            (output["dinov2_blend_label"] != output["resnet101_final_label"]).sum()
        ),
        "conservative_changed_count": int(output["dinov2_changed"].sum()),
        "primary_submission": "submission.csv",
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    (WORK_DIR / "dinov2_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    log(json.dumps(summary, indent=2))
    gc.collect()


run()
